In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv langchain-community langchain-core langchain langchain-openai langchain-chroma

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드 (.env 파일에는 OPENAI API 키값을 적으면 됩니다. -> OPENAI_API_KEY=...)
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
# <이전 대화를 포함한 메시지 전달>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 금융 상담 역할
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        ("placeholder", "{messages}"),  # 대화 이력 추가
    ]
)

# 프롬프트와 모델을 연결하여 체인 생성
chain = prompt | chat

# 이전 대화를 포함한 메시지 전달
ai_msg = chain.invoke(
    {
        "messages": [
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"),  # 사용자의 첫 질문
            ("ai", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),  # 챗봇의 답변
            ("human", "방금 뭐라고 했나요?"),  # 사용자의 재확인 질문
        ],
    }
)
print(ai_msg.content)  # 챗봇의 응답 출력


저축을 늘리기 위해 목표를 설정하고 매달 자동 이체로 일정 금액을 저축하는 것을 추천했습니다. 이렇게 하면 저축 습관을 형성할 수 있고, 금액이 자동으로 저축 계좌로 이체되어 관리하기 더 쉬워집니다. 추가로 다음과 같은 방법들도 고려해 보세요:

1. **예산 수립**: 수입과 지출을 명확히 파악해 보세요. 이를 통해 절약할 수 있는 부분을 찾아 저축으로 돌릴 수 있습니다.
   
2. **비상금 마련**: 예기치 않은 지출에 대비해 일정 금액을 비상금으로 마련해 두세요. 이렇게 하면 저축이 줄어드는 것을 방지할 수 있습니다.

3. **불필요한 지출 줄이기**: 구독 서비스나 다소 불필요한 소비를 검토하고 줄여보세요.

4. **할인 및 세일 활용**: 할인 행사나 세일을 통해 필요한 물건을 구입하면 절약할 수 있습니다.

5. **저축 목표 구체화**: 예를 들어, 여행, 주택 구매, 긴급 자금 등 구체적인 저축 목표를 세우면 동기부여가 됩니다.

이런 방법들을 통해 저축을 점차 증가시킬 수 있을 것입니다.


In [7]:
# <`ChatMessageHistory`를 사용한 메시지 관리>

from langchain_community.chat_message_histories import ChatMessageHistory

# 대화 이력 저장을 위한 클래스 초기화
chat_history = ChatMessageHistory()

# 사용자 메시지 추가
chat_history.add_user_message("저축을 늘리기 위해 무엇을 할 수 있나요?")
chat_history.add_ai_message("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요.")

# 새로운 질문 추가 후 다시 체인 실행
chat_history.add_user_message("방금 뭐라고 했나요?")
ai_response = chain.invoke({"messages": chat_history.messages})
print(ai_response.content)  # 챗봇은 이전 메시지를 기억하여 답변합니다.

저축을 늘리기 위해 저축 목표를 세우고, 매달 자동 이체로 일정 금액을 저축하는 것이 좋다고 말씀드렸습니다. 이렇게 하면 저축을 꾸준히 늘릴 수 있습니다. 추가적으로, 필요하지 않은 지출을 줄이고, 예산을 세워 관리하는 것도 도움이 됩니다.


In [8]:
# <` RunnableWithMessageHistory`를 사용한 메시지 관리>

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 시스템 메시지와 대화 이력을 사용하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)

# 대화 이력을 관리할 체인 설정
chat_history = ChatMessageHistory()
chain = prompt | chat

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# 질문 메시지 체인 실행
chain_with_message_history.invoke(
    {"input": "저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable": {"session_id": "unused"}},
).content


d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'저축을 늘리기 위해 여러 가지 방법을 고려할 수 있습니다. 다음은 효과적인 몇 가지 방법입니다:\n\n1. **예산 세우기**: 매달 수입과 지출을 정리하여 예산을 세우세요. 이를 통해 고정비와 변동비를 파악하고, 불필요한 지출을 줄인 후 남는 돈을 저축에 돌릴 수 있습니다.\n\n2. **자동 저축 설정**: 월급이 들어오면 자동으로 일정 금액을 저축 계좌로 이체하도록 설정해 보세요. 이렇게 하면 저축을 잊지 않고 지속적으로 할 수 있습니다.\n\n3. **비상금 마련**: 예상치 못한 지출에 대비할 수 있는 비상금을 마련해 두는 것도 좋습니다. 이는 긴급 상황에서 저축이 줄어드는 것을 방지합니다.\n\n4. **지출 관리 앱 활용**: 다양한 앱을 사용하여 지출을 추적하고 관리하세요. 이를 통해 어디에서 돈을 많이 쓰는지 분석하고, 조정할 수 있습니다.\n\n5. **소비 습관 변화**: 필요 없는 소비를 줄이고, 할인 쿠폰이나 세일을 활용하여 지출을 절약하는 습관을 기릅니다. 예를 들어, 외식 대신 집에서 요리하는 것도 도움이 됩니다.\n\n6. **저축 목표 설정**: 단기 및 장기 저축 목표를 설정하고, 목표에 도달하기 위한 계획을 세워 동기를부여합니다.\n\n7. **추가 수입 창출**: 본업 외에 프리랜스 일거리나 아르바이트, 투자 등을 통해 추가 수입을 올려 저축을 늘릴 수 있습니다.\n\n이러한 방법들을 통해 저축을 효과적으로 늘릴 수 있을 것입니다. 또한, 저축을 위해 금융 상품에 대해 공부하고 적절한 저축 방법을 선택하는 것도 중요합니다.'

In [9]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content


'당신은 "저축을 늘리기 위해 무엇을 할 수 있나요?"라고 질문하셨습니다. 저축을 늘리기 위한 여러 방법에 대해 설명드렸습니다. 추가적인 질문이나 도움이 필요하시면 언제든지 말씀해 주세요!'

In [10]:
# <메시지 트리밍 예제>

# 라이브러리 불러오기
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# 메시지 트리밍 유틸리티 설정
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | chat
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 새로운 대화 내용 추가 후 체인 실행
chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
)



AIMessage(content='5년 내에 집을 사기 위한 재정 계획을 세우는 것은 중요한 목표입니다. 아래는 그 목표를 달성하기 위한 단계적인 계획입니다:\n\n1. **예산 수립**:\n   - 월 소득과 지출을 분석하여 현재 재정 상황을 파악합니다.\n   - 필수 지출과 비상금 외에 주택 구매를 위한 저축에 얼마를 할당할 수 있는지 결정합니다.\n\n2. **구매 목표 설정**:\n   - 원하는 집의 가격 범위를 설정합니다. 지역에 따라 가격이 크게 다를 수 있으므로 여러 지역의 부동산 시장을 조사합니다.\n   - 초기 계약금(일반적으로 10-20%)과 추가 비용(소비세, 변호사 비용, 보험 등)을 포함하여 총 구매 비용을 예측합니다.\n\n3. **저축 목표 설정**:\n   - 목표 집 가격의 20%를 계약금으로 저축하기 위해 필요한 월별 저축액을 계산합니다.\n   - 예를 들어, 3억원의 집을 구입한다고 가정할 때, 20%인 6천만원을 5년(60개월) 동안 저축하려면 매월 약 100만원을 저축해야 합니다.\n\n4. **재정 상품 활용하고 투자**:\n   - 고수익 저축 계좌, 적금, 혹은 안전한 투자 옵션을 선택하여 저축한 자금을 늘릴 수 있도록 합니다.\n   - 위험 감수 정도에 따라 주식이나 펀드에 일부 자금을 투자할 수도 있습니다.\n\n5. **부채 관리**:\n   - 기존의 부채가 있다면 우선적으로 그 부채를 줄이는 방향으로 재정 계획을 세웁니다.\n   - 신용 점수를 관리하여 향후 대출을 받을 때 유리한 조건을 확보합니다.\n\n6. **재정 상담 받기**:\n   - 필요시 전문가와 상담하여 재정 계획을 더욱 세부적으로 조정한 후 포함해야 할 사항을 점검합니다.\n\n7. **주택 구매 관련 정보 조사**:\n   - 주택 구매 시 혜택을 받을 수 있는 정부 지원 프로그램도 조사하여 필요한 경우 지원을 받습니다.\n\n계획적으로 저축하고 투자하기 위해 설정한 목표와 예산을 주기적으로 점검하는 것이 중요합니다. 이렇게 하

In [11]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
).content


'당신은 "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"라고 질문하셨습니다. 이 질문에 대해 재정 계획 수립을 위한 단계적인 접근 방식을 제공했습니다. 더 궁금한 점이나 추가적인 질문이 있다면 말씀해 주세요!'

In [12]:
# <이전 대화 요약 내용 기반으로 답변하기>

def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False
    # 대화를 요약하기 위한 프롬프트 템플릿 설정
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),  # 이전 대화 이력
            (
                "user",
                "이전 대화를 요약해 주세요. 가능한 한 많은 세부 정보를 포함하십시오.",  # 요약 요청 메시지
            ),
        ]
    )

    # 요약 체인 생성 및 실행
    summarization_chain = summarization_prompt | chat
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})
    chat_history.clear()  # 요약 후 이전 대화 삭제
    chat_history.add_message(summary_message)  # 요약된 메시지를 대화 이력에 추가
    return True

In [13]:
# 대화 요약을 처리하는 체인 설정
chain_with_summarization = (
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_summarization.invoke(
    {"input": "저에게 어떤 재정적 조언을 해주셨나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content)


앞서 제안한 재정적 조언은 대략 다음과 같습니다:

1. **저축을 늘리기 위한 방법**:
   - **예산 세우기**: 월 수입과 지출을 정리하여 불필요한 지출을 줄이는 것이 중요합니다.
   - **자동 저축 설정**: 월급이 들어오면 자동으로 일정 금액을 저축 계좌로 이체하는 방식으로 저축을 지속적으로 할 수 있습니다.
   - **비상금 마련**: 예상치 못한 지출에 대비할 수 있는 비상금을 마련하는 것이 좋습니다.
   - **소비 습관 변화**: 할인 쿠폰이나 세일을 활용하여 불필요한 소비를 줄입니다.
   - **저축 목표 설정**: 단기 및 장기 저축 목표를 설정하여 동기를 부여합니다.
   - **추가 수입 창출**: 본업 외에 추가 수입을 올릴 수 있는 기회를 찾아 저축을 늘립니다.

2. **5년 내에 집을 사기 위한 재정 계획**:
   - **예산 수립**: 월 소득과 지출을 분석하여 주택 구매를 위한 저축 가능 금액을 결정합니다.
   - **구매 목표 설정**: 원하는 집의 가격, 계약금 및 추가 비용을 예측합니다.
   - **저축 목표 설정**: 목표 집 가격의 계약금을 5년 동안 저축하기 위한 월별 저축액을 계산합니다.
   - **부채 관리**: 기존 부채를 줄이고 신용 점수를 관리하여 주택 구매에 유리한 조건을 만드는 것이 좋습니다.
   - **재정 상담 받기**: 전문가와 상담하여 더 구체적인 계획을 세웁니다.

이 조언들은 저축과 재정 계획을 효과적으로 수행하는 데 도움을 줄 수 있습니다. 더 구체적인 질문이 있으시거나 추가 도움이 필요하시면 언제든지 말씀해 주세요!
